# boop: train your own small model (free, Kaggle T4)

**Settings:** Accelerator **GPU T4 x1** · Internet **On** · Secret **`HF_TOKEN`** (write).
Needs the dataset from `data/kaggle_data.ipynb` on HF first.

1. `MAX_STEPS = 30` for a 5-minute smoke test, then `-1` for the real run (~15–25 min for 360M).
2. The last cells score each GGUF on the 34 eval tasks (Linux/bash here). Plain Qwen 1.5B scored **79%** on your Mac: that's the bar.

In [ ]:
# ---- config ----------------------------------------------------------------------------
HF_REPO   = "bunnyweb/boop-360m-gguf"            # where the GGUF files go ("" = don't push)
BASE      = "HuggingFaceTB/SmolLM2-360M-Instruct"
EPOCHS    = 3
MAX_STEPS = 30                                  # 30 = smoke test; -1 = full run
GIT_REPO  = "https://github.com/thebunnyweb/boop"
DATA_HF   = "bunnyweb/boop-data"

In [ ]:
import os, subprocess, glob
WORK = "/kaggle/working/boop"
if not os.path.exists(WORK):
    subprocess.run(["git", "clone", "-q", GIT_REPO, WORK], check=True)
os.chdir(WORK)
!git pull -q && git log --oneline -1
!pip install -q unsloth
!pip install -q -e . --no-deps
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
from huggingface_hub import snapshot_download
d = snapshot_download(DATA_HF, repo_type="dataset", allow_patterns=["master/*"], token=os.environ["HF_TOKEN"])
os.makedirs("data/out", exist_ok=True)
!rm -rf data/out/master && cp -r {d}/master data/out/master && cat data/out/master/stats.json

In [ ]:
push = f"--push {HF_REPO}" if HF_REPO else ""
!python training/train.py --data data/out/master --out runs/boop --base {BASE} \
    --epochs {EPOCHS} --max-steps {MAX_STEPS} --gguf q8_0,q4_k_m {push}

## Score it on the eval (same 34 tasks as on your Mac)

In [ ]:
!pip install -q "llama-cpp-python>=0.3.0" --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu
GGUFS = sorted(g for g in glob.glob("runs/boop/**/*.gguf", recursive=True) if "Q8_0" in g.upper() or "Q4_K_M" in g.upper())
for g in GGUFS:
    print(f"\n===== {os.path.basename(g)}  ({os.path.getsize(g)/1e6:.0f} MB)")
    !BOOP_MODEL={g} BOOP_FEWSHOT=0 python eval/run_eval.py --shell bash --dump eval/out/{os.path.basename(g)}.jsonl

In [ ]:
# try it yourself
g = GGUFS[0]
for q in ["find pdfs over 10mb in downloads", "kill whatever is on port 3000", "how many lines are in app.log",
          "zip the src folder", "undo my last commit but keep the changes"]:
    !BOOP_MODEL={g} BOOP_FEWSHOT=0 boop -p --shell bash "{q}"